In [ ]:
import lightning as L
import torch
from IPython.display import Video

from training.evaluate import evaluate, record_video
from training.ppo import ActorCritic, PolicyModule, RolloutData, flat, normalize
from training.sim import SKILLS, DuckEnv, hardware

In [ ]:
hw = hardware.load("2026-11-26-hd1910")
skill = SKILLS["course-walk"]
env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()                           # 回合进度打散，和正式训练一样
net = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")   # 61 个观测进，14 个关节的动作出

In [ ]:
def gae(rewards, dones, values, next_value, gamma, lam):
    advantages = torch.zeros_like(rewards)
    running = torch.zeros_like(next_value)
    for t in reversed(range(len(rewards))):              # 从最后一步往前算
        next_values = next_value if t == len(rewards) - 1 else values[t + 1]
        not_done = 1 - dones[t]                          # 回合在这一步结束就截断
        delta = rewards[t] + gamma * next_values * not_done - values[t]   # 时序差分误差
        running = delta + gamma * lam * not_done * running
        advantages[t] = running
    return advantages, advantages + values               # 优势，价值目标

In [ ]:
class A2CData(RolloutData):                              # data 一半
    gamma, lam = 0.99, 0.95

    def prepare(self, rollout):
        values = rollout["values"].squeeze(-1)
        rewards = rollout["rewards"] + self.gamma * values * rollout["time_outs"]   # 超时只是截断：补上折扣后的估值
        advantages, returns = gae(rewards, rollout["dones"], values, rollout["next_value"].squeeze(-1),
                                  self.gamma, self.lam)
        return {**flat(rollout, "obs", "critic_obs", "actions"),
                "advantages": normalize(advantages).flatten(), "returns": returns.flatten()}

In [ ]:
class A2C(PolicyModule):                                 # model 一半
    entropy_coef = 0.01
    value_loss_coef = 1.0

    def loss(self, batch):
        dist = self.net.action_distribution(batch["obs"])
        log_probs = dist.log_prob(batch["actions"]).sum(-1)
        entropy = dist.entropy().sum(-1).mean()
        values = self.net.value(batch["critic_obs"]).squeeze(-1)
        policy_loss = -(log_probs * batch["advantages"]).mean()
        value_loss = (values - batch["returns"]).pow(2).mean()      # critic 的估值往价值目标靠
        return policy_loss + self.value_loss_coef * value_loss - self.entropy_coef * entropy

In [ ]:
before = evaluate(net, skill, hw,
                  common_step_counter=env.unwrapped.common_step_counter)   # 512 只鸭子各走 20 秒
before["falls"], round(before["mean_step_reward"], 3)

In [ ]:
# 每轮重建一次数据，也就是每轮用最新的网络重新采样
trainer = L.Trainer(max_epochs=1500, reload_dataloaders_every_n_epochs=1,
                    accelerator="gpu", devices=1, logger=False, enable_checkpointing=False)
trainer.fit(A2C(net), A2CData(env, net))

In [ ]:
after = evaluate(net, skill, hw,
                 common_step_counter=env.unwrapped.common_step_counter)
after["falls"], round(after["mean_step_reward"], 3)

In [ ]:
v1 = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim)
v1.load_state_dict(torch.load("../S03-policy-gradient/v1_reinforce.pt"))
reinforce = evaluate(v1, skill, hw,
                     common_step_counter=env.unwrapped.common_step_counter)
reinforce["falls"], round(reinforce["mean_step_reward"], 3)

In [ ]:
record_video(net, skill, hw, "v2_a2c.mp4",
             common_step_counter=env.unwrapped.common_step_counter)
Video("v2_a2c.mp4", embed=True)

In [ ]:
torch.save(net.state_dict(), "v2_a2c.pt")